In [ ]:
import warnings
import numpy as np
import pandas as pd

from sklearn.preprocessing import LabelEncoder, StandardScaler

from pgmpy.estimators import HillClimbSearch, BicScore
from pgmpy.models import BayesianNetwork
from pgmpy.inference import VariableElimination

warnings.filterwarnings("ignore")

/Users/lucas/anaconda3/envs/mineria/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
df = pd.read_csv("Students Social Media Addiction.csv")

df = df.drop("Student_ID", axis=1)

cat_cols = df.select_dtypes(include="object").columns
le = LabelEncoder()
for col in cat_cols:
    df[col] = le.fit_transform(df[col])

X = df.drop("Mental_Health_Score", axis=1)
y = df["Mental_Health_Score"]

scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

In [ ]:
df_for_causal = X_scaled.copy()
df_for_causal["Mental_Health_Score"] = y.values

df_discrete = df_for_causal.copy()

for col in df_discrete.columns:
    if df_discrete[col].dtype in ['float64', 'int64']:
        df_discrete[col] = pd.qcut(df_discrete[col], q=3, labels=False, duplicates='drop')

hc = HillClimbSearch(df_discrete)
best_model = hc.estimate(scoring_method=BicScore(df_discrete))

print("Learned structure edges:")
print(best_model.edges())

model = BayesianNetwork(best_model.edges())
model.fit(df_discrete)

inference = VariableElimination(model)

q = inference.query(variables=["Mental_Health_Score"], evidence={"Addicted_Score": 1})
print(q)

  0%|          | 12/1000000 [00:00<13:09:32, 21.11it/s]


Learned structure edges:
[('Age', 'Most_Used_Platform'), ('Avg_Daily_Usage_Hours', 'Addicted_Score'), ('Most_Used_Platform', 'Country'), ('Most_Used_Platform', 'Sleep_Hours_Per_Night'), ('Sleep_Hours_Per_Night', 'Avg_Daily_Usage_Hours'), ('Sleep_Hours_Per_Night', 'Addicted_Score'), ('Sleep_Hours_Per_Night', 'Mental_Health_Score'), ('Relationship_Status', 'Country'), ('Conflicts_Over_Social_Media', 'Mental_Health_Score'), ('Conflicts_Over_Social_Media', 'Relationship_Status'), ('Conflicts_Over_Social_Media', 'Country'), ('Addicted_Score', 'Conflicts_Over_Social_Media')]
+------------------------+----------------------------+
| Mental_Health_Score    |   phi(Mental_Health_Score) |
+========================+============================+
| Mental_Health_Score(0) |                     0.8322 |
+------------------------+----------------------------+
| Mental_Health_Score(1) |                     0.1516 |
+------------------------+----------------------------+
| Mental_Health_Score(2) |      

In [6]:
print(inference.query(variables=["Mental_Health_Score"], evidence={"Addicted_Score": 2}))

+------------------------+----------------------------+
| Mental_Health_Score    |   phi(Mental_Health_Score) |
+========================+============================+
| Mental_Health_Score(0) |                     0.9981 |
+------------------------+----------------------------+
| Mental_Health_Score(1) |                     0.0015 |
+------------------------+----------------------------+
| Mental_Health_Score(2) |                     0.0004 |
+------------------------+----------------------------+


In [7]:
print(inference.query(variables=["Mental_Health_Score"], evidence={"Addicted_Score": 2, "Sleep_Hours_Per_Night": 0}))

+------------------------+----------------------------+
| Mental_Health_Score    |   phi(Mental_Health_Score) |
+========================+============================+
| Mental_Health_Score(0) |                     1.0000 |
+------------------------+----------------------------+
| Mental_Health_Score(1) |                     0.0000 |
+------------------------+----------------------------+
| Mental_Health_Score(2) |                     0.0000 |
+------------------------+----------------------------+


In [8]:
print(inference.query(variables=["Mental_Health_Score"], evidence={"Addicted_Score": 0}))

+------------------------+----------------------------+
| Mental_Health_Score    |   phi(Mental_Health_Score) |
+========================+============================+
| Mental_Health_Score(0) |                     0.0000 |
+------------------------+----------------------------+
| Mental_Health_Score(1) |                     0.5755 |
+------------------------+----------------------------+
| Mental_Health_Score(2) |                     0.4245 |
+------------------------+----------------------------+
